# Deepo — Machine Translation System
**Seq2Seq LSTM | Anglais → 10 langues**

Sections :
1. Setup & Google Drive
2. Extraction des données
3. Nettoyage des données
4. Ajout des alias de langue
5. Construction du CSV
6. Entraînement du modèle
7. Inférence (traduction)

## 1. Setup & Google Drive

In [10]:
from google.colab import drive
drive.mount('/content/drive')

!pip install tqdm -q

import os

# Détecte automatiquement le dossier où est le notebook
BASE      = os.getcwd()
DATA_DIR  = os.path.join(BASE, 'datasets')
ZIP_DIR   = os.path.join(BASE, 'zip_data')
RAW_DIR   = os.path.join(BASE, 'raw_data')
CLEAN_DIR = os.path.join(BASE, 'clean_data')
CKPT_PATH = '/content/drive/MyDrive/best.pt'

for d in [ZIP_DIR, RAW_DIR, CLEAN_DIR, DATA_DIR]:
    os.makedirs(d, exist_ok=True)

print('BASE     :', BASE)
print('DATA_DIR :', DATA_DIR)
print('CKPT_PATH:', CKPT_PATH)
print('Setup OK')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE     : /content
DATA_DIR : /content/datasets
CKPT_PATH: /content/drive/MyDrive/best.pt
Setup OK


## 2. Extraction des données
Upload les **8 fichiers zip** suivants dans `/content/zip_data/` via le panneau fichiers à gauche :
- `fra-eng.zip`, `deu-eng.zip`, `spa-eng.zip`, `por-eng.zip`
- `ara-eng.zip`, `jpn-eng.zip`, `cmn-eng.zip`, `bre-eng.zip`

Ensuite lance cette cellule.

In [22]:
import zipfile

KEEP_LANGS = {'fra', 'deu', 'spa', 'por', 'ara', 'jpn', 'cmn', 'bre'}

for fname in os.listdir(ZIP_DIR):
    if not fname.endswith('.zip'):
        continue
    lang = fname.replace('-eng.zip', '')
    if lang not in KEEP_LANGS:
        continue
    with zipfile.ZipFile(os.path.join(ZIP_DIR, fname)) as z:
        for name in z.namelist():
            if name.endswith('.txt') and not name.startswith('_about'):
                z.extract(name, RAW_DIR)
                print(f'Extrait : {name}')

print('Extraction terminée')

Extrait : fra.txt
Extraction terminée


## 3. Nettoyage des données
Supprime la colonne licence, garde uniquement `source\tcible`.

In [23]:
for filename in os.listdir(RAW_DIR):
    if not filename.endswith('.txt'):
        continue

    src_path = os.path.join(RAW_DIR, filename)
    dst_path = os.path.join(CLEAN_DIR, filename)

    with open(src_path, 'r', encoding='utf-8') as src, \
         open(dst_path, 'w', encoding='utf-8') as dst:

        for line in src:
            line = line.strip()
            if not line:
                continue
            parts = line.split('\t')
            if len(parts) >= 2:
                dst.write(f'{parts[0].strip()}\t{parts[1].strip()}\n')

    print(f'Nettoyé : {filename}')

print('Nettoyage terminé')

Nettoyé : fra.txt
Nettoyage terminé


## 4. Ajout des alias de langue
Préfixe chaque phrase source avec le tag `>>lang<<` (ex: `>>fra<<`).

In [24]:
for filename in os.listdir(CLEAN_DIR):
    if not filename.endswith('.txt'):
        continue

    lang = os.path.splitext(filename)[0]
    tag  = f'>>{lang}<< '
    path = os.path.join(CLEAN_DIR, filename)

    with open(path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    new_lines = []
    for line in lines:
        line = line.rstrip('\n')
        if not line.strip():
            continue
        parts = line.split('\t')
        if len(parts) != 2:
            continue
        src, tgt = parts
        new_lines.append(f'{tag}{src.strip()}\t{tgt.strip()}\n')

    with open(path, 'w', encoding='utf-8') as f:
        f.writelines(new_lines)

    print(f'Alias ajouté : {filename}')

print('Alias terminés')

Alias ajouté : fra.txt
Alias terminés


## 5. Construction du CSV
Filtre, échantillonne 40% de chaque langue, et découpe en train/valid/test (90/5/5).

In [25]:
import glob, random, csv, re

PERCENT = 1.0
random.seed(42)

def normalize_src(sentence):
    # Supprime la ponctuation de la source (anglais) → "I love you!" → "I love you"
    sentence = re.sub(r"[?.!,;:]", " ", sentence)
    return re.sub(r" +", " ", sentence).strip()

def normalize_tgt(sentence):
    # Espace autour de la ponctuation cible pour la tokenisation → "Bonjour." → "Bonjour ."
    sentence = re.sub(r"([?.!,;:'])", r" \1 ", sentence)
    return re.sub(r" +", " ", sentence).strip()

def ok_pair(src, tgt):
    src, tgt = src.strip(), tgt.strip()
    if not src or not tgt: return False
    if len(src.split()) < 1 or len(tgt.split()) < 1: return False
    if len(src.split()) > 25 or len(tgt.split()) > 25: return False
    return True

pairs = []

for path in glob.glob(os.path.join(CLEAN_DIR, '*.txt')):
    lang_pairs = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip('\n').split('\t')
            if len(parts) != 2: continue
            src, tgt = parts
            src = normalize_src(src)
            tgt = normalize_tgt(tgt)
            if ok_pair(src, tgt):
                lang_pairs.append((src, tgt))
    pairs.extend(lang_pairs)
    print(f'{os.path.basename(path)} → {len(lang_pairs)} paires')

random.shuffle(pairs)
n = len(pairs)
n_train = int(0.90 * n)
n_valid = int(0.05 * n)

def write_csv(name, data):
    out = os.path.join(DATA_DIR, f'{name}.csv')
    with open(out, 'w', encoding='utf-8', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['source', 'target'])
        writer.writerows(data)
    print(f'Écrit : {out} ({len(data)} lignes)')

write_csv('train', pairs[:n_train])
write_csv('valid', pairs[n_train:n_train + n_valid])
write_csv('test',  pairs[n_train + n_valid:])
print(f'\nTotal : {n} paires')

fra.txt → 238798 paires
Écrit : /content/datasets/train.csv (214918 lignes)
Écrit : /content/datasets/valid.csv (11939 lignes)
Écrit : /content/datasets/test.csv (11941 lignes)

Total : 238798 paires


## 6. Entraînement du modèle
Modèle **Seq2Seq LSTM** — Encoder / Decoder avec Teacher Forcing.

In [ ]:
import csv, math, random
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt

# --- Config ---
SEED            = 42
DEVICE          = 'cuda' if torch.cuda.is_available() else 'cpu'
MAX_LEN         = 60
MIN_FREQ        = 5
BATCH           = 512
EMB             = 512
HID             = 512
EPOCHS          = 25
LR              = 1e-3
TEACHER_FORCING = 0.6
PAD, SOS, EOS, UNK = '<pad>', '<s>', '</s>', '<unk>'

random.seed(SEED)
torch.manual_seed(SEED)
print('DEVICE:', DEVICE)

# --- Utilitaires ---
def tokenize(s):
    s = s.strip()
    # CJK (chinois, japonais, coréen) → découpe caractère par caractère
    if any('\u3000' <= c <= '\u9fff' or '\uac00' <= c <= '\ud7af' for c in s):
        return list(s.replace(' ', ''))
    return s.split()

def read_csv_pairs(path):
    pairs = []
    with open(path, 'r', encoding='utf-8', newline='') as f:
        for row in csv.DictReader(f):
            src = (row.get('source') or '').strip()
            tgt = (row.get('target') or '').strip()
            if src and tgt:
                pairs.append((src, tgt))
    return pairs

class Vocab:
    def __init__(self, texts, min_freq=1):
        counter = Counter()
        for t in texts:
            counter.update(tokenize(t))
        self.itos = [PAD, SOS, EOS, UNK]
        for w, c in counter.items():
            if c >= min_freq and w not in self.itos:
                self.itos.append(w)
        self.stoi = {w: i for i, w in enumerate(self.itos)}
        self.pad = self.stoi[PAD]
        self.sos = self.stoi[SOS]
        self.eos = self.stoi[EOS]
        self.unk = self.stoi[UNK]

    def encode(self, s, add_sos=False, add_eos=False):
        ids = []
        if add_sos: ids.append(self.sos)
        for tok in tokenize(s)[:MAX_LEN]:
            ids.append(self.stoi.get(tok, self.unk))
        if add_eos: ids.append(self.eos)
        return ids

    def __len__(self):
        return len(self.itos)

class PairsDataset(Dataset):
    def __init__(self, pairs, src_vocab, tgt_vocab):
        self.pairs = pairs
        self.sv = src_vocab
        self.tv = tgt_vocab

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        src, tgt = self.pairs[idx]
        return (
            torch.tensor(self.sv.encode(src, add_eos=True), dtype=torch.long),
            torch.tensor(self.tv.encode(tgt, add_sos=True, add_eos=True), dtype=torch.long)
        )

def collate_fn(batch, src_pad, tgt_pad):
    srcs, tgts = zip(*batch)
    src_lens = torch.tensor([len(x) for x in srcs], dtype=torch.long)
    tgt_lens = torch.tensor([len(x) for x in tgts], dtype=torch.long)
    src_batch = torch.full((len(batch), src_lens.max()), src_pad, dtype=torch.long)
    tgt_batch = torch.full((len(batch), tgt_lens.max()), tgt_pad, dtype=torch.long)
    for i, (s, t) in enumerate(zip(srcs, tgts)):
        src_batch[i, :len(s)] = s
        tgt_batch[i, :len(t)] = t
    return src_batch, src_lens, tgt_batch, tgt_lens

# --- Attention de Bahdanau ---
class Attention(nn.Module):
    def __init__(self, hid_dim):
        super().__init__()
        self.attn = nn.Linear(hid_dim * 2, hid_dim)
        self.v    = nn.Linear(hid_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        B, S, H = encoder_outputs.shape
        hidden  = hidden.permute(1, 0, 2).repeat(1, S, 1)
        energy  = torch.tanh(self.attn(torch.cat([hidden, encoder_outputs], dim=2)))
        return torch.softmax(self.v(energy).squeeze(2), dim=1)

# --- Modèle ---
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, pad_id):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.rnn = nn.LSTM(emb_dim, hid_dim, batch_first=True)

    def forward(self, src):
        outputs, (h, c) = self.rnn(self.emb(src))
        return outputs, h, c

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, pad_id):
        super().__init__()
        self.emb  = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.attn = Attention(hid_dim)
        self.rnn  = nn.LSTM(emb_dim + hid_dim, hid_dim, batch_first=True)
        self.fc   = nn.Linear(hid_dim, vocab_size)

    def forward(self, inp, h, c, encoder_outputs):
        embedded     = self.emb(inp).unsqueeze(1)
        attn_weights = self.attn(h, encoder_outputs).unsqueeze(1)
        context      = torch.bmm(attn_weights, encoder_outputs)
        rnn_input    = torch.cat([embedded, context], dim=2)
        out, (h, c)  = self.rnn(rnn_input, (h, c))
        return self.fc(out.squeeze(1)), h, c

class Seq2Seq(nn.Module):
    def __init__(self, enc, dec, device):
        super().__init__()
        self.enc    = enc
        self.dec    = dec
        self.device = device

    def forward(self, src, tgt, teacher_forcing=0.5):
        B, T = tgt.size()
        V    = self.dec.fc.out_features
        outputs           = torch.zeros(B, T, V, device=self.device)
        enc_outputs, h, c = self.enc(src)
        inp = tgt[:, 0]
        for t in range(1, T):
            logits, h, c  = self.dec(inp, h, c, enc_outputs)
            outputs[:, t] = logits
            top1 = logits.argmax(1)
            inp  = tgt[:, t] if random.random() < teacher_forcing else top1
        return outputs

# --- Boucle d'entraînement ---
def run_epoch(model, loader, optim, crit, train=True, ep=0):
    model.train(train)
    total_loss    = 0.0
    correct_toks  = 0
    total_toks    = 0
    label = 'train' if train else 'valid'
    bar   = tqdm(loader, desc=f'epoch {ep} [{label}]', unit='batch')

    for src, src_lens, tgt, tgt_lens in bar:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        if train: optim.zero_grad()

        out    = model(src, tgt, teacher_forcing=TEACHER_FORCING if train else 0.0)
        logits = out[:, 1:].reshape(-1, out.size(-1))
        gold   = tgt[:, 1:].reshape(-1)
        loss   = crit(logits, gold)

        if train:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()

        mask          = gold != tgt_vocab.pad
        preds         = logits.argmax(1)
        correct_toks += (preds[mask] == gold[mask]).sum().item()
        total_toks   += mask.sum().item()

        total_loss += loss.item()
        bar.set_postfix(loss=f'{loss.item():.4f}')

    avg_loss = total_loss / max(1, len(loader))
    accuracy = correct_toks / max(1, total_toks) * 100
    return avg_loss, accuracy

# --- Main ---
train_pairs = read_csv_pairs(os.path.join(DATA_DIR, 'train.csv'))
valid_pairs = read_csv_pairs(os.path.join(DATA_DIR, 'valid.csv'))

src_vocab = Vocab([s for s, _ in train_pairs], min_freq=MIN_FREQ)
tgt_vocab = Vocab([t for _, t in train_pairs], min_freq=MIN_FREQ)
print(f'Vocab source : {len(src_vocab)} | Vocab cible : {len(tgt_vocab)}')

train_ds = PairsDataset(train_pairs, src_vocab, tgt_vocab)
valid_ds = PairsDataset(valid_pairs, src_vocab, tgt_vocab)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
    collate_fn=lambda b: collate_fn(b, src_vocab.pad, tgt_vocab.pad))
valid_loader = DataLoader(valid_ds, batch_size=BATCH, shuffle=False,
    collate_fn=lambda b: collate_fn(b, src_vocab.pad, tgt_vocab.pad))

enc   = Encoder(len(src_vocab), EMB, HID, src_vocab.pad)
dec   = Decoder(len(tgt_vocab), EMB, HID, tgt_vocab.pad)
model = Seq2Seq(enc, dec, DEVICE).to(DEVICE)
optim = torch.optim.Adam(model.parameters(), lr=LR)
crit  = nn.CrossEntropyLoss(ignore_index=tgt_vocab.pad)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optim, patience=3, factor=0.5)

history = {'train_loss': [], 'valid_loss': [], 'train_acc': [], 'valid_acc': []}

best = float('inf')
for ep in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(model, train_loader, optim, crit, train=True,  ep=ep)
    va_loss, va_acc = run_epoch(model, valid_loader, optim, crit, train=False, ep=ep)
    ppl = math.exp(min(va_loss, 10))

    scheduler.step(va_loss)

    history['train_loss'].append(tr_loss)
    history['valid_loss'].append(va_loss)
    history['train_acc'].append(tr_acc)
    history['valid_acc'].append(va_acc)

    print(f'epoch {ep:>2} | train loss {tr_loss:.4f} | valid loss {va_loss:.4f} | ppl {ppl:.2f} | train acc {tr_acc:.1f}% | valid acc {va_acc:.1f}%')

    if va_loss < best:
        best = va_loss
        torch.save({
            'model':    model.state_dict(),
            'src_itos': src_vocab.itos,
            'tgt_itos': tgt_vocab.itos,
            'emb': EMB, 'hid': HID
        }, CKPT_PATH)
        print(f'  Sauvegardé → {CKPT_PATH}')

# --- Visualisation ---
epochs_range = range(1, len(history['train_loss']) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs_range, history['train_loss'], 'b-o', label='Train Loss')
ax1.plot(epochs_range, history['valid_loss'], 'r-o', label='Valid Loss')
ax1.set_title('Loss par epoch')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

ax2.plot(epochs_range, history['train_acc'], 'b-o', label='Train Accuracy')
ax2.plot(epochs_range, history['valid_acc'], 'r-o', label='Valid Accuracy')
ax2.set_title('Accuracy par epoch')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/training_curves.png', dpi=150)
plt.show()
print('Graphique sauvegardé sur Drive !')

## 7. Inférence — Traduction
Charge le meilleur modèle et traduit une phrase.

In [ ]:
import os, re, torch, torch.nn as nn

DEVICE  = 'cuda' if torch.cuda.is_available() else 'cpu'
MAX_LEN = 60
PAD, SOS, EOS, UNK = '<pad>', '<s>', '</s>', '<unk>'

LOCAL_CKPT = os.path.join(os.getcwd(), 'models', 'lstm_seq2seq', 'best.pt')
DRIVE_CKPT = '/content/drive/MyDrive/best.pt'
CKPT_PATH  = LOCAL_CKPT if os.path.exists(LOCAL_CKPT) else DRIVE_CKPT
print('Chargement depuis :', CKPT_PATH)

class Attention(nn.Module):
    def __init__(self, hid_dim):
        super().__init__()
        self.attn = nn.Linear(hid_dim * 2, hid_dim)
        self.v    = nn.Linear(hid_dim, 1, bias=False)
    def forward(self, hidden, encoder_outputs):
        B, S, H = encoder_outputs.shape
        hidden  = hidden.permute(1, 0, 2).repeat(1, S, 1)
        energy  = torch.tanh(self.attn(torch.cat([hidden, encoder_outputs], dim=2)))
        return torch.softmax(self.v(energy).squeeze(2), dim=1)

class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, pad_id):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.rnn = nn.LSTM(emb_dim, hid_dim, batch_first=True)
    def forward(self, src):
        outputs, (h, c) = self.rnn(self.emb(src))
        return outputs, h, c

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, pad_id):
        super().__init__()
        self.emb  = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.attn = Attention(hid_dim)
        self.rnn  = nn.LSTM(emb_dim + hid_dim, hid_dim, batch_first=True)
        self.fc   = nn.Linear(hid_dim, vocab_size)
    def forward(self, inp, h, c, encoder_outputs):
        embedded     = self.emb(inp).unsqueeze(1)
        attn_weights = self.attn(h, encoder_outputs).unsqueeze(1)
        context      = torch.bmm(attn_weights, encoder_outputs)
        rnn_input    = torch.cat([embedded, context], dim=2)
        out, (h, c)  = self.rnn(rnn_input, (h, c))
        return self.fc(out.squeeze(1)), h, c

LANGUAGES = {
    '1': ('Français',  'fra'),
    '2': ('Espagnol',  'spa'),
    '3': ('Allemand',  'deu'),
    '4': ('Portugais', 'por'),
    '5': ('Arabe',     'ara'),
    '6': ('Japonais',  'jpn'),
    '7': ('Chinois',   'cmn'),
    '8': ('Breton',    'bre'),
}

def tokenize(s):
    s = s.strip()
    # CJK (chinois, japonais, coréen) → découpe caractère par caractère
    if any('\u3000' <= c <= '\u9fff' or '\uac00' <= c <= '\ud7af' for c in s):
        return list(s.replace(' ', ''))
    return s.split()

def load_model(ckpt_path):
    ck       = torch.load(ckpt_path, map_location=DEVICE)
    src_itos = ck['src_itos']
    tgt_itos = ck['tgt_itos']
    src_stoi = {w: i for i, w in enumerate(src_itos)}
    tgt_stoi = {w: i for i, w in enumerate(tgt_itos)}
    emb, hid = ck['emb'], ck['hid']
    enc = Encoder(len(src_itos), emb, hid, src_stoi[PAD])
    dec = Decoder(len(tgt_itos), emb, hid, tgt_stoi[PAD])
    full = ck['model']
    enc.load_state_dict({k[4:]: v for k, v in full.items() if k.startswith('enc.')})
    dec.load_state_dict({k[4:]: v for k, v in full.items() if k.startswith('dec.')})
    enc.to(DEVICE).eval()
    dec.to(DEVICE).eval()
    return enc, dec, src_stoi, tgt_itos, tgt_stoi

def preprocess(sentence):
    # Supprime la ponctuation de la source — cohérent avec l'entraînement
    sentence = re.sub(r"[?.!,;:]", " ", sentence)
    return re.sub(r" +", " ", sentence).strip()

def postprocess(tokens):
    tokens = [t for t in tokens if t != UNK]
    text = ' '.join(tokens)
    # Recolle les apostrophes : "J ' aime" → "J'aime"
    text = re.sub(r" ' ", "'", text)
    # Recolle la ponctuation : "Bonjour ." → "Bonjour."
    text = re.sub(r" ([.!?,;:])", r"\1", text)
    # Pour le CJK : supprime les espaces entre caractères chinois/japonais
    text = re.sub(r'(?<=[\u4e00-\u9fff\u3040-\u30ff]) (?=[\u4e00-\u9fff\u3040-\u30ff])', '', text)
    return text if text else '(modèle pas encore assez entraîné)'

def translate(sentence, enc, dec, src_stoi, tgt_itos, tgt_stoi):
    clean = preprocess(sentence)
    ids = [src_stoi.get(t, src_stoi[UNK]) for t in tokenize(clean)]
    ids.append(src_stoi[EOS])
    src_tensor = torch.tensor(ids, dtype=torch.long).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        enc_outputs, h, c = enc(src_tensor)
        inp = torch.tensor([tgt_stoi[SOS]], dtype=torch.long).to(DEVICE)
        result = []
        for _ in range(MAX_LEN):
            logits, h, c = dec(inp, h, c, enc_outputs)
            top1 = logits.argmax(1).item()
            if tgt_itos[top1] == EOS: break
            result.append(tgt_itos[top1])
            inp = torch.tensor([top1], dtype=torch.long).to(DEVICE)
    return postprocess(result)

enc, dec, src_stoi, tgt_itos, tgt_stoi = load_model(CKPT_PATH)
print('Modèle chargé !\n')
for k, (name, _) in LANGUAGES.items():
    print(f'  {k}. {name}')

In [29]:
import ipywidgets as widgets
from IPython.display import display, clear_output

sentence_input = widgets.Text(
    placeholder='Ex: I love Paris.',
    description='Anglais :',
    layout=widgets.Layout(width='500px')
)

lang_dropdown = widgets.Dropdown(
    options=[(name, code) for _, (name, code) in LANGUAGES.items()],
    description='Langue :',
)

btn = widgets.Button(description='Traduire', button_style='primary')
output = widgets.Output()

def on_click(b):
    with output:
        clear_output()
        sentence = sentence_input.value.strip()
        if not sentence:
            print("Écris une phrase !")
            return
        code      = lang_dropdown.value
        lang_name = lang_dropdown.label
        src_with_tag = f'>>{code}<< {sentence.capitalize()}'
        translation  = translate(src_with_tag, enc, dec, src_stoi, tgt_itos, tgt_stoi)
        print(f"Source  : {sentence}")
        print(f"Langue  : {lang_name}")
        print(f"Traduit : {translation}")

btn.on_click(on_click)
display(sentence_input, lang_dropdown, btn, output)

Text(value='', description='Anglais :', layout=Layout(width='500px'), placeholder='Ex: I love Paris.')

Dropdown(description='Langue :', options=(('Français', 'fra'), ('Espagnol', 'spa'), ('Allemand', 'deu'), ('Por…

Button(button_style='primary', description='Traduire', style=ButtonStyle())

Output()